# MediNest Fine-Tuning
Llama 3.2 3B → medical triage specialist

**Before running:** Upload `training_data.jsonl` to this Colab session (Files panel on the left → Upload)

In [ ]:
# Cell 1 — Install (takes ~3 min)
!pip install unsloth -q
!pip install --upgrade --quiet trl transformers datasets peft accelerate bitsandbytes

In [ ]:
# Cell 2 — Load base model (Llama 3.2 3B, 4-bit quantized)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)
print("Model loaded")

In [ ]:
# Cell 3 — Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA applied")

In [ ]:
# Cell 4 — Load your dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

def format_messages(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_messages)
print(f"Dataset: {len(dataset)} examples")
print("Sample:", dataset[0]["text"][:300])

In [ ]:
# Cell 5 — Train (takes ~45 min on free T4)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=42,
    ),
)

trainer.train()

In [ ]:
# Cell 6 — Export to GGUF (format Ollama understands)
# This saves medinest-model.Q4_K_M.gguf (~2 GB)
model.save_pretrained_gguf("medinest-model", tokenizer, quantization_method="q4_k_m")
print("Done! Download medinest-model.Q4_K_M.gguf from the Files panel")

In [ ]:
# Cell 7 — (Optional) Download directly via Google Drive instead of Files panel
# Useful if the file is too large to download directly
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy("medinest-model.Q4_K_M.gguf", "/content/drive/MyDrive/medinest-model.Q4_K_M.gguf")
print("Saved to Google Drive")

## After downloading the .gguf file

1. Put `medinest-model.Q4_K_M.gguf` in your `fine-tuning/` folder
2. Run this in your terminal:
```
ollama create medinest-llama3 -f Modelfile-finetuned
```
That's it — your app will automatically use the fine-tuned model.